In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Differentiable Simulations with NVIDIA Warp

## Overview

Having built a forward Navier–Stokes solver in Notebook 1, we now make it **differentiable**, meaning we can compute gradients of simulation outputs with respect to simulation inputs efficiently and automatically.

Why would we want this? A forward solver answers: given these initial conditions, what flow develops? Many interesting problems, however, run in the opposite direction: instead of predicting outcomes from inputs, we ask what inputs produce a desired outcome. These include inverse problems, where we seek to recover unknown parameters from observed data, as well as design and optimization problems, where we search for inputs that minimize or maximize a chosen objective. With a differentiable solver, we can back-propagate a scalar loss through every timestep to obtain exact gradients, turning the simulation itself into a differentiable function that plugs directly into gradient-based optimizers.

This same capability also enables **hybrid ML + physics** workflows: a neural network can propose initial conditions or closure terms, the differentiable solver rolls them forward while respecting the governing equations, and the combined pipeline can be trained end-to-end. The network learns from the physics, and the physics constrains the network.

In this notebook, we begin with two optimization examples that use the differentiable solver:

- Finding an initial vorticity field that evolves into the NVIDIA GTC logo
- Finding a perturbation to an initial vorticity field that maximizes divergence between the perturbed and unperturbed trajectories

We then unpack the three Warp features that make these examples possible:

- No in-place modifications of intermediate states
- Recording gradients with ``wp.Tape()``
- Closing the optimization loop with ``warp.optim``


<img src="./images/differentiable-navier-stokes/optimization_simulation_compressed.gif" alt="Turbulence simulation preview" width="400">

In [ ]:
# List the version of Warp installed in the current environment
!pip list | grep -E "warp-lang"

# List the version of other libraries installed in the current environment
!pip list | grep -E "matplotlib|ipympl|numpy"

# Shorten exception tracebacks for readability in this notebook
%xmode Minimal

In [ ]:
import numpy as np
import warp as wp

if wp.get_cuda_device_count() > 0:
    print("GPU detected successfully")
else:
    print("No GPU detected!")

---
## Example 1: Optimizing an initial condition to match a target field

<div align="center">
  <img src="./images/differentiable-navier-stokes/gtc_logo.png" width="200">
</div>

In this example, we make our forward solver differentiable and use it to solve an inverse problem: given a target vorticity field at a lead time $T$, find the initial vorticity field $\omega_0$ that evolves into it. Here, the target field, denoted $\omega_{\mathrm{NVIDIA}}$, is the vorticity image constructed from the NVIDIA GTC logo.

Let $\mathcal{F}^{T}(\omega_0)$ denote the vorticity field obtained by advancing $\omega_0$ through $T$ time-steps of the Navier–Stokes solver. The loss is the mean-squared error computed over all image pixels between the evolved field and the target image:

$$ \mathrm{Loss} = \mathrm{MSE}(\omega_{\mathrm{NVIDIA}},\; \mathcal{F}^{T}(\omega_0)).$$

Because the loss depends on $\omega_0$ through every intermediate time-step, minimizing it requires gradients that backpropagate through the entire simulation. This is precisely what a differentiable solver provides.

To keep this example focused on the optimization loop, we package the inverse-problem setup in ``VorticityInverseSolver()``. The class bundles the differentiable solver rollout, loss computation, and optimizer update. ``LEAD_STEPS`` sets the forward horizon $T$, and ``OPTIMIZER_STEPS`` sets the number of optimization iterations. For readers who want the implementation details, see ``helpers/differentiable_navier_stokes/logo_solver.py``. Feel free to vary these values and see how the recovered initial condition changes.

In [ ]:
# ======================== CONTROLS ========================
LEAD_STEPS = 80       # Lead time T (simulation timesteps)
OPTIMIZER_STEPS = 150      # Optimization iterations
# ==========================================================

from helpers.differentiable_navier_stokes.logo_solver import VorticityInverseSolver

example = VorticityInverseSolver(
    target_image_path="./images/differentiable-navier-stokes/gtc_logo.png",
    lead_steps=LEAD_STEPS,
)

In [ ]:
losses = []
snapshots = []

for i in range(OPTIMIZER_STEPS):
    loss_val = example.train_step(LEAD_STEPS)
    losses.append(loss_val)

    if i % 10 == 0 or i == OPTIMIZER_STEPS - 1:
        snapshots.append((
            i,
            example.omega_init.numpy().copy(),
            example.omega_timestep[LEAD_STEPS].numpy().copy(),
        ))
        print(f"Iter {i:4d} | Loss: {loss_val:.6f}")

In [ ]:
from helpers.differentiable_navier_stokes.utils import save_logo_optimization_gif
from IPython.display import Image

save_logo_optimization_gif(snapshots, losses)
Image(filename="outputs/logo_optimization.gif")

As optimization progresses, the initial vorticity field is updated so that, after $T$ time-steps, the evolved vorticity field matches the GTC logo target more closely.

---
## Example 2: Optimal initial perturbation

Using our differentiable solver, we now solve an optimal perturbation problem: find the initial perturbation $\Delta\omega$ to a base field $\omega_0$ that maximizes the separation between the perturbed and unperturbed vorticity fields at time $T$. The problem is inspired by the work of [Kim et al. (2024)](https://www.cambridge.org/core/journals/journal-of-fluid-mechanics/article/prediction-and-control-of-twodimensional-decaying-turbulence-using-generative-adversarial-networks/EAE377A4E1F784D135520DB1EE3F25A0). Identifying which perturbations grow the fastest is a stepping stone toward flow control and toward understanding which structures in the flow are dynamically significant.

Let $\mathcal{F}^T$ denote the forward operator that evolves an initial vorticity field to time $T$. We seek

$$
\max_{\|\Delta\omega\| \leq \varepsilon} \;
\text{MSE}\!\bigl(\mathcal{F}^T(\omega_0),\;
\mathcal{F}^T(\omega_0 + \Delta\omega)\bigr).
$$

We solve this constrained optimization problem by minimizing the negative MSE with gradient descent, while requiring the perturbation field's L2 norm to remain below a prescribed threshold.

Similar to the previous problem, the problem setup is packaged in ``OptimalPerturbationSolver()``. ``LEAD_STEPS`` sets the forward horizon $T$ and ``OPTIMIZER_STEPS`` sets the number of optimizer iterations. For readers who want the implementation details, see ``helpers/differentiable_navier_stokes/perturbation_solver.py``.

In [ ]:
# ======================== CONTROLS ========================
LEAD_STEPS = 80        # Lead time T (simulation timesteps)
OPTIMIZER_STEPS = 100       # Optimization iterations
# ==========================================================

from helpers.differentiable_navier_stokes.perturbation_solver import OptimalPerturbationSolver

example = OptimalPerturbationSolver(lead_steps=LEAD_STEPS)

In [ ]:
losses = []
snapshots = []

for i in range(OPTIMIZER_STEPS):
    loss_val = example.train_step(LEAD_STEPS)
    losses.append(loss_val)

    if i % 10 == 0 or i == OPTIMIZER_STEPS - 1:
        snapshots.append((
            i,
            example.delta_omega.numpy().copy(),
            example.omega_timesteps[LEAD_STEPS].numpy().copy(),
        ))
        print(f"Iter {i:4d} | Loss: {loss_val:.6f}")

In [ ]:
from helpers.differentiable_navier_stokes.utils import save_perturbation_optimization_gif
from IPython.display import Image

save_perturbation_optimization_gif(snapshots, losses)
Image(filename="outputs/perturbation_optimization.gif")

As the optimization progresses, structured patterns emerge in the initial perturbation field (middle figure). Those perturbations then stretch and rotate the vorticity field at lead time $T$.

We encourage you to explore the [technical blog](https://developer.nvidia.com/blog/build-accelerated-differentiable-computational-physics-code-for-ai-with-nvidia-warp/) and the [example code](https://github.com/NVIDIA/warp/blob/main/warp/examples/optim/example_navier_stokes_perturbation.py) for a deeper look at the optimal-perturbation formulation and its full implementation in Warp.

---
## Key changes relative to a non-differentiable solver

The optimization examples above show what a differentiable solver can enable. In this section, we focus on the three implementation changes needed to make the Navier-Stokes solver differentiable in Warp:

- No in-place modification of intermediate states
- Recording the forward pass with ``wp.Tape()``
- Closing the optimization loop with ``warp.optim``

These same changes apply to other differentiable Warp programs as well. In particular, Warp autodiff requires all input and intermediate arrays that participate in differentiation to be explicitly allocated and managed.

### No in-place modifications

Warp has a context manager, ``wp.Tape()``, that records kernel launches during the forward pass and replays them in reverse to compute gradients. In reverse mode AD, we must preserve intermediate arrays from the forward pass so they are available during the backward pass. Unlike PyTorch and JAX, which implicitly allocate new memory for intermediate results, Warp requires us to manage that memory explicitly. A useful rule of thumb is: **don't write to arrays you have read from**.

As a simple example, suppose we want to evaluate the expression $y = x^2 + 3x + 1$ and compute its gradient, $\mathrm{d}y/\mathrm{d}x$.

In Warp, if we write the result back into ``x[tid]``, we should not expect the correct gradient:

```python
@wp.kernel
def kernel_func(x: wp.array(dtype=float)):
    tid = wp.tid()
    x[tid] = x[tid] ** 2.0 + 3.0 * x[tid] + 1.0

x = wp.array([1.0, 2.0, 3.0], dtype=float)
```

Instead, we need to allocate a separate output array ``y`` and write the result there:

```python
@wp.kernel
def kernel_func(x: wp.array(dtype=float), y: wp.array(dtype=float)):
    tid = wp.tid()
    y[tid] = x[tid] ** 2.0 + 3.0 * x[tid] + 1.0

x = wp.array([1.0, 2.0, 3.0], dtype=float, requires_grad=True)
y = wp.zeros_like(x)
```

``wp.Tape()`` can also visualize the recorded computation graph via ``Tape.visualize()``, producing a GraphViz ``dot`` file that shows how arrays flow between kernel launches. For the non-fused N-body example from ``00_Warp_Intro.ipynb``, this makes the dependencies between ``compute_accelerations``, ``update_velocities``, and ``update_positions`` explicit, which is helpful when debugging missing ``requires_grad=True`` flags or incorrect array flow.

<div style="background-color:white; text-align:center;">
    <img src="./images/differentiable-navier-stokes/nbody_nonfused_tape_graph_simplified_horizontal.png" width="700">
</div>

A small but important detail is that graph visualization works best when ``wp.launch()`` uses explicit ``inputs=[...]`` and ``outputs=[...]`` lists, rather than passing output arrays through ``inputs=[...]``.

This is the key difference from the non-differentiable solver. In our forward-only version, the state arrays ``omega_0`` and ``omega_1`` could be overwritten at the end of each timestep as follows:

```python
wp.copy(omega_0, omega_1)
```

![SSA vs in-place](./images/differentiable-navier-stokes/ssa_vs_inplace.svg)

For both examples above, we can no longer rely on implicit intermediates. In Warp, arrays whose values must participate in the backward pass need to be allocated explicitly, and arrays whose gradients we want to accumulate must be created with ``requires_grad=True``. In practice, this usually includes the optimized inputs, the loss array, and intermediate arrays whose values must be preserved for reverse-mode AD.

When this flag is set, Warp also allocates a companion ``.grad`` array where the corresponding adjoint values are stored.

For Runge-Kutta time integration, storing one state per timestep is not enough. Each RK stage applies additional intermediate updates within the same timestep, and the backward pass needs those stage-level values as well, not just the final state at $t^{n+1}$.

For example, ``helpers/differentiable_navier_stokes/perturbation_solver.py`` defines the following class method to allocate all intermediate arrays with ``requires_grad=True``.

```python
    def _allocate_per_step_arrays(self) -> None:
        """Pre-allocate arrays for the differentiable forward pass.

        Warp's autodiff requires each operation to write to a unique array so
        that gradients can be replayed in reverse.  We therefore allocate
        separate omega, psi, and FFT-scratch arrays for every (timestep, RK stage)
        pair.
        """
        T = self.lead_steps

        self.omega_timesteps = [wp.zeros((N_GRID, N_GRID), dtype=float, requires_grad=True) for _ in range(T + 1)]

        # Per-stage arrays for each timestep, indexed as [timestep][stage].
        self.omega_ts = []
        self.psi_ts = []
        self.fft_ts = []

        for _ in range(T):
            stage_omega, stage_psi, stage_fft = [], [], []
            for _ in range(3):
                stage_omega.append(wp.zeros((N_GRID, N_GRID), dtype=float, requires_grad=True))
                stage_psi.append(wp.zeros((N_GRID, N_GRID), dtype=float, requires_grad=True))
                stage_fft.append(self._allocate_poisson_fft_buffers(requires_grad=True))
            self.omega_ts.append(stage_omega)
            self.psi_ts.append(stage_psi)
            self.fft_ts.append(stage_fft)
```

Storing intermediate arrays for reverse-mode AD comes with a memory cost that grows linearly with the number of time steps, i.e. $O(T)$. Since each saved state already scales with the number of spatial degrees of freedom, this can become prohibitively expensive for long-time simulations.

A common remedy is **gradient checkpointing**: save only selected states during the forward pass, then recompute the missing segments when running the backward pass. This trades additional compute for a substantially smaller memory footprint. Gradient checkpointing is also used in [deep learning](https://arxiv.org/pdf/1604.06174) to trade extra compute for lower memory usage.

For an example of gradient checkpointing in Warp, see the [fluid checkpoint example](https://github.com/NVIDIA/warp/blob/main/warp/examples/optim/example_fluid_checkpoint.py) in the NVIDIA/warp repository.

### Recording the forward pass with ``wp.Tape()``

``wp.Tape()`` captures the forward computational graph described above. Once that graph is stored in the ``tape`` object, calling ``tape.backward(loss)`` traverses it in reverse and computes gradients such as $\partial \mathcal{L} / \partial x_i$ for all recorded inputs and intermediate arrays.

Looking again at ``helpers/differentiable_navier_stokes/perturbation_solver.py``, we find the following snippet showing how the forward pass is recorded and then differentiated:

```python
self.tape = wp.Tape()
with self.tape:
    self.forward()
self.tape.backward(self.loss)
```

After the backward pass, every array created with ``requires_grad=True`` has its adjoint stored in the corresponding ``.grad`` array.

### Closing the optimization loop with ``warp.optim``

The final piece of the puzzle is closing the optimization loop. Consider an abstract optimization problem in which we want to update $\mathbf{x}$ so that $\mathcal{L}$ decreases:

$$
\mathcal{L} = f(\mathbf{x}).
$$

As we saw above, Warp's autodiff machinery lets us compute $\partial \mathcal{L} / \partial x_i$ by calling ``tape.backward(loss)``. Once these derivatives are available, the next step is to update $\mathbf{x}$ so as to minimize $\mathcal{L}$.

Taken together, the partial derivatives $\partial \mathcal{L} / \partial x_i$ form the gradient $\nabla \mathcal{L}$, which points in the direction of steepest increase of $\mathcal{L}$ at $\mathbf{x}$. To decrease the loss, we therefore update $\mathbf{x}$ in the direction of $-\nabla \mathcal{L}$.

As in PyTorch and other deep learning frameworks, Warp provides the ``wp.optim`` module, which includes gradient-based optimizers such as Adam and SGD. Once the forward and backward passes are in place, the update syntax is straightforward. In ``helpers/differentiable_navier_stokes/perturbation_solver.py``, the following snippets show how the optimizer is initialized for the vorticity perturbation and how a single optimization step is taken.

```python
self.optimizer = warp.optim.Adam([self.delta_omega.flatten()], lr=lr)

# Compute x.grad after evaluating loss and calling tape.backward(loss)
self.optimizer.step([self.delta_omega.grad.flatten()])
```

---
## Interoperability with other DL frameworks

Finally, let's look at a concrete example of how Warp's autodifferentiation capabilities can interoperate with other DL frameworks. To motivate this, we briefly discuss the work of [Ma et al. (2023)](https://arxiv.org/pdf/2304.14369#page=13.56), who used a hybrid neural network and PDE solver approach to discover constitutive laws (relationships between stress and deformation).

<img src="./images/differentiable-navier-stokes/million_particle_compressed.gif" alt="MPM preview" width="400">

The key takeaway here is how naturally Warp and PyTorch complement each other. The differentiable solver was built in Warp, which made it possible to take advantage of kernel fusion and Warp's built-in data structures for simulation workloads, while the neural network components were built in PyTorch. Because Warp interoperates cleanly with frameworks such as PyTorch and JAX, gradients can flow across the solver-ML boundary while each part of the pipeline stays in the tool where it is strongest.

<div align="center">
  <img src="./images/differentiable-navier-stokes/ma-et-al-solver-in-loop.png" width="800">
</div>

The snippet below is schematic pseudocode that shows how gradients can flow across the PyTorch-Warp boundary. The exact kernel signature, launch dimension, and saved context depend on the solver you are wrapping.

```python
class SolverFunction(torch.autograd.Function):
    # Forward: PyTorch -> Warp -> PyTorch

    @staticmethod
    def forward(ctx, inputs):
        tape = wp.Tape()
        warp_in = wp.from_torch(inputs)  # zero-copy into Warp
        warp_out = wp.zeros_like(warp_in)

        with tape:
            wp.launch(kernel, dim=warp_in.shape, inputs=[warp_in], outputs=[warp_out])

        ctx.tape = tape
        ctx.warp_in = warp_in
        ctx.warp_out = warp_out
        return wp.to_torch(warp_out)  # zero-copy back to PyTorch

    # Backward: gradients flow PyTorch -> Warp -> PyTorch

    @staticmethod
    def backward(ctx, grad_outputs):
        ctx.warp_out.grad = wp.from_torch(grad_outputs)
        ctx.tape.backward()
        return wp.to_torch(ctx.warp_in.grad)
```

---
## Conclusion

In this notebook, we turned our Navier-Stokes solver into a differentiable simulation pipeline. We learned:

- How Warp's autodiff model requires us to explicitly preserve the intermediate states needed for backpropagation
- Why making a solver differentiable requires avoiding in-place updates, recording the forward pass with ``wp.Tape()``, and using ``wp.optim`` to close the optimization loop
- How these pieces enable inverse problems and trajectory optimization by propagating exact gradients through the full simulation rollout
- How Warp can interoperate with frameworks such as PyTorch and JAX, letting us combine high-performance solvers with the broader neural network ecosystem

Taken together, these features make Warp a strong foundation for hybrid physics + ML workflows, where simulation and learning can be optimized end-to-end within a single differentiable pipeline.

--- 
## References

* Optimal perturbation in a 2-D Navier-Stokes solver example in Warp, https://github.com/NVIDIA/warp/blob/main/warp/examples/optim/example_navier_stokes_perturbation.py
* Documentation for differentiability in Warp, GitHub Pages, https://nvidia.github.io/warp/stable/user_guide/differentiability.html
* Documentation for interoperability in Warp, GitHub Pages, https://nvidia.github.io/warp/stable/user_guide/interoperability.html
* "Learning Neural Constitutive Laws From Motion Observations for Generalizable PDE Dynamics", https://arxiv.org/pdf/2304.14369#page=13.56
* "Prediction and control of two-dimensional decaying turbulence using generative adversarial networks", https://www.cambridge.org/core/journals/journal-of-fluid-mechanics/article/prediction-and-control-of-twodimensional-decaying-turbulence-using-generative-adversarial-networks/EAE377A4E1F784D135520DB1EE3F25A0
